# AI Resume Screener - Step by Step Pipeline

This notebook runs the AI Resume Screener pipeline component by component, printing the intermediate output at each step so we can verify exactly what the system "thinks" and extracts.

In [1]:
# 1. Setup & Imports
import os
import sys

# Ensure we can import the backend package
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from backend.pdf_parser import extract_text_from_pdf, extract_links_from_pdf
from backend.anonymizer_agent import redact_pii, extract_structured_profile
from backend.github_scraper import fetch_github_profile, extract_github_username
from backend.codeforces_scraper import fetch_codeforces_profile, extract_codeforces_handle
from backend.embeddings import embed_text, embed_codes
from backend.scorer import compute_final_score
from backend.explainer import generate_explanation

import json
from pprint import pprint

print("✅ Setup complete. All backend modules imported.")

✅ Setup complete. All backend modules imported.


### Step 1: Parse the PDF
First, we extract the raw text and any clickable hyperlinks stored in the PDF annotations.

In [7]:
# CHANGE THIS path to match your actual resume PDF file path
RESUME_PATH = r"C:\Users\SUDARSHAN\Downloads\resume_sudarshan_kulkarni (1).pdf"  # Replace with actual absolute path if needed
# RESUME_PATH = r"C:\Users\SUDARSHAN\Downloads\abhyudaya_singh_resume.pdf"

# RESUME_PATH = r"C:\Users\SUDARSHAN\Downloads\11005406.pdf" #1 
# RESUME_PATH = r"C:\Users\SUDARSHAN\Downloads\11270462.pdf" #2
# RESUME_PATH = r"C:\Users\SUDARSHAN\Downloads\11677012.pdf" #3
# RESUME_PATH = r"C:\Users\SUDARSHAN\Downloads\10515955.pdf" #4
# RESUME_PATH = r"C:\Users\SUDARSHAN\Downloads\12085736.pdf" #5


try:
    print("--- STEP 1: PDF EXTRACTION ---\n")
    raw_text = extract_text_from_pdf(RESUME_PATH)
    pdf_links = extract_links_from_pdf(RESUME_PATH)
    
    print(f"Extracted Text Length: {len(raw_text)} characters")
    print("\nExtracted Hyperlinks from PDF Annotations:")
    for i, link in enumerate(pdf_links):
        print(f"  {i+1}. {link}")
        
    print("\nText Snippet (first 400 chars):")
    print("-" * 40)
    print(raw_text + "...")
    print("-" * 40)
except FileNotFoundError:
    print(f"❌ Could not find {RESUME_PATH}. Please update the path above!")

--- STEP 1: PDF EXTRACTION ---

Extracted Text Length: 3845 characters

Extracted Hyperlinks from PDF Annotations:
  1. https://www.linkedin.com/in/sudarshan-k-7a1a35233/
  2. https://github.com/AMOGHA1140
  3. https://github.com/AMOGHA1140/english_hindi_translation
  4. https://github.com/AMOGHA1140/24074032_CSOC_IG/blob/main/vanilla_neural_networks/main.py
  5. https://github.com/AMOGHA1140/24074032_CSOC_IG/tree/main/CSGO_model
  6. https://www.coursera.org/account/accomplishments/specialization/OMDM83TKS2HB?utm_source=link&utm_medium=certificate&utm_content=cert_image&utm_campaign=sharing_cta&utm_product=s12n
  7. https://www.coursera.org/account/accomplishments/specialization/FGXYC499JWK5
  8. https://drive.google.com/file/d/1Jg0YMkY7FUn9zrxSObuEchaK7b2NCncX/view?usp=drive_link
  9. https://drive.google.com/file/d/1x1IJi2fgyYg7sNk4wtk17J4UH2BwvIuL/view?usp=drive_link

Text Snippet (first 400 chars):
----------------------------------------
Sudarshan Kulkarni
+91 99672 15737 |amoghs

### Step 2: Redact PII (Bias Prevention)
We use Gemini to replace names, locations, and other identifying information with `[Redacted]`.

In [8]:
print("--- STEP 2: PII REDACTION ---\n")
print("Calling Gemini to redact PII (this might take a few seconds)...\n")
redacted_text = redact_pii(raw_text)

print("Redacted Text Snippet (first 400 chars):")
print("-" * 40)
print(redacted_text[:400] + "...")
print("-" * 40)

--- STEP 2: PII REDACTION ---

Calling Gemini to redact PII (this might take a few seconds)...

Redacted Text Snippet (first 400 chars):
----------------------------------------
[Name Redacted]
[Phone Redacted] | [Email Redacted] | Github

Education
[University Redacted], [Location Redacted] Aug. 2024 – Jul. 2029
B.Tech and M.Tech in Computer Science Current CPI: 9.42/10.0

Experience
InterIIT Tech Meet 14.0 Nov 2025 – Dec 2025
[Company Redacted] Bronze Medalist
•Engineered a Multimodal vision-language framework for Natural Language Interpretation of Satellite Imagery, se...
----------------------------------------


### Step 3: Extract Structured Profile
We ask Gemini to extract skills, work history, and URLs into a pure JSON schema, passing along the `pdf_links` to ensure no URLs are missed.

In [9]:
print("--- STEP 3: STRUCTURED EXTRACTION ---\n")
print("Extracting skills, projects, and external URLs...\n")

profile = extract_structured_profile(redacted_text, pdf_links=pdf_links)

print("✅ Profile Extracted. Here is what the system sees:\n")
print(json.dumps(profile, indent=2))

--- STEP 3: STRUCTURED EXTRACTION ---

Extracting skills, projects, and external URLs...

✅ Profile Extracted. Here is what the system sees:

{
  "skills": [
    "Python",
    "C",
    "C++",
    "PyTorch",
    "HuggingFace",
    "Pandas",
    "NumPy",
    "Matplotlib",
    "Git",
    "Github",
    "LaTeX",
    "Cursor",
    "CNN",
    "LSTM",
    "Transformers",
    "ViTs",
    "VAE",
    "MoE"
  ],
  "projects": [
    {
      "name": "English to Hindi Translation",
      "description": "Engineered a Neural Machine Translation model from scratch in PyTorch, implementing the original Transformer architecture and training on multiple Kaggle English-Hindi datasets.",
      "tech_stack": [
        "Python",
        "Pytorch"
      ],
      "url": "https://github.com/AMOGHA1140/english_hindi_translation"
    },
    {
      "name": "Neural Network Library",
      "description": "Architected a modular Neural Network framework from scratch in Python and NumPy, designing a dynamic API to build

### Step 4: Scrape GitHub & Codeforces
Now we take the extracted handles and hit the APIs to gather evidence.

In [10]:
print("--- STEP 4: EXTERNAL SCRAPING ---\n")

github_username = profile.get("github_username")
codeforces_handle = profile.get("codeforces_handle")

github_data = {"code_signals": {}, "verified_dependencies": [], "top_repos": []}
codeforces_data = None
has_github = False

if github_username!="null":
    print(f"🐙 Found GitHub username: {github_username}")
    print("Scraping GitHub (repos, dependencies, code files)...\n")
    github_data = fetch_github_profile(github_username)
    if github_data.get("error"):
        print(f"⚠️ GitHub Error: {github_data['error']}")
    else:
        has_github = True
        print(f"✅ Fetched {len(github_data['top_repos'])} repos, {len(github_data['verified_dependencies'])} dependencies.")
        print(f"✅ Found code files for {len(github_data['code_signals'])} top repos.\n")
else:
    print("ℹ️ No GitHub profile found.")

if codeforces_handle!="null":
    print(f"🏆 Found Codeforces handle: {codeforces_handle}")
    codeforces_data = fetch_codeforces_profile(codeforces_handle)
    if codeforces_data.get("error"):
        print(f"⚠️ Codeforces Error: {codeforces_data['error']}")
    else:
        print(f"✅ Max Rating: {codeforces_data['max_rating']}, Solved ≈ {codeforces_data['solved_problems_approx']}")
else:
    print("ℹ️ No Codeforces profile found.")

--- STEP 4: EXTERNAL SCRAPING ---

🐙 Found GitHub username: AMOGHA1140
Scraping GitHub (repos, dependencies, code files)...

✅ Fetched 5 repos, 2 dependencies.
✅ Found code files for 3 top repos.

ℹ️ No Codeforces profile found.


In [11]:
github_data

{'username': 'AMOGHA1140',
 'public_repos': 6,
 'followers': 1,
 'following': 0,
 'top_repos': [{'name': '24074032_CSOC_IG',
   'stars': 0,
   'forks': 0,
   'language': 'Jupyter Notebook',
   'description': '',
   'topics': []},
  {'name': 'ctrl_shift_intelligence',
   'stars': 0,
   'forks': 1,
   'language': 'Python',
   'description': '',
   'topics': []},
  {'name': 'english_hindi_translation',
   'stars': 0,
   'forks': 0,
   'language': 'Jupyter Notebook',
   'description': '',
   'topics': []},
  {'name': 'LLM_Council_Academic_Usage',
   'stars': 0,
   'forks': 0,
   'language': 'Python',
   'description': '',
   'topics': []},
  {'name': 'mnist_classifier',
   'stars': 0,
   'forks': 0,
   'language': 'Jupyter Notebook',
   'description': '',
   'topics': []}],
 'language_distribution': {'Jupyter Notebook': 60, 'Python': 40},
 'verified_dependencies': ['google-genai', 'pydantic'],
 'raw_dependency_files': {'LLM_Council_Academic_Usage/requirements.txt': 'google-genai\npydantic'

### Step 5: Deterministic Scoring
We calculate the mathematical score. We need a Job Description to compare against.

In [12]:
print("--- STEP 5: MATHEMATICAL SCORING ---\n")

JOB_DESCRIPTION = """
We are looking for an AI research intern having experience in previous research projects.
The experience with Python, PyTorch and knowledge of Machine Learning in depth is necessary. 
"""

# JOB_DESCRIPTION = "A leading pharmaceutical company committed to developing and commercializing innovative and high-quality medicines that improve the lives of patients is now hiring for a Senior Product Marketing Manager. This role will be responsible for developing and executing marketing strategies for one or more pharmaceutical products. This individual will work closely with cross-functional teams including sales, medical affairs, market research, and commercial operations to develop and implement integrated marketing plans that drive product awareness, adoption, and revenue growth."

# Collect code files for evidence matching
code_contents = []
for repo_name, signals in github_data.get("code_signals", {}).items():
    for f in signals.get("code_files", []):
        content = f.get("content", "")
        if content.strip():
            code_contents.append(content)
    readme = signals.get("readme")
    if readme:
        code_contents.append(readme)

print(f"Calculating embeddings... (comparing against {len(code_contents)} code files for evidence)")

scoring_result = compute_final_score(
    jd_text=JOB_DESCRIPTION,
    resume_text=redacted_text,
    code_contents=code_contents,
    claimed_skills=profile.get("skills", []),
    verified_deps=github_data.get("verified_dependencies", []),
    work_history=profile.get("work_history", []),
    has_github_evidence=has_github,
    codeforces_data=codeforces_data,
    jd_requires_cp="competitive programming" in JOB_DESCRIPTION.lower()
)

print("\n✅ Scoring Complete!")
print(f"FINAL SCORE: {scoring_result.final_score:.4f} / 1.0")
print("\nBreakdown:")
pprint(scoring_result.component_breakdown)
print(f"\nVerified Skills: {scoring_result.verified_skills}")
print(f"Unverified Skills: {scoring_result.unverified_skills}")

--- STEP 5: MATHEMATICAL SCORING ---

Calculating embeddings... (comparing against 9 code files for evidence)
[-0.03076386  0.02438547  0.00559695 ... -0.01249792  0.01730771
  0.00194591]
[[-0.01320321  0.02551087  0.01673046 ... -0.00750009 -0.00276279
   0.01071187]
 [-0.00663108  0.03596475  0.01838264 ...  0.00769252 -0.02094961
  -0.00190424]
 [-0.00013419  0.02410823 -0.01382365 ... -0.01432757 -0.00169872
  -0.01509304]
 ...
 [-0.01639748  0.00925386  0.00643455 ...  0.0134059  -0.00032697
   0.01302547]
 [-0.02770351  0.02231524  0.0156678  ... -0.01001679  0.00087309
  -0.01151728]
 [-0.01369458  0.02225056  0.001931   ... -0.00755447 -0.01197199
  -0.00379102]]

✅ Scoring Complete!
FINAL SCORE: 0.5238 / 1.0

Breakdown:
{'cp_bonus': 0.0,
 'evidence_match': {'value': 0.575901, 'weight': 0.3, 'weighted': 0.17277},
 'experience_signal': {'value': 0.826667, 'weight': 0.15, 'weighted': 0.124},
 'semantic_match': {'value': 0.746536, 'weight': 0.3, 'weighted': 0.223961},
 'verificat

### Step 6: Glass-box Explanation Generation
Finally, we use Gemini Pro to generate human-readable pros, cons, and evidence chains that explain the math.

In [13]:
print("--- STEP 6: EXPLANATION ENGINE ---\n")
print("Calling Gemini to explain the score...\n")

explanation = generate_explanation(
    scoring_result=scoring_result,
    jd_text=JOB_DESCRIPTION,
    github_data=github_data,
    codeforces_data=codeforces_data,
    work_history=profile.get("work_history", [])
)

print("✅ Explanation Generated:\n")
print("SUMMARY:")
print(explanation.get("summary", ""))
print("\nPROS:")
for p in explanation.get("pros", []):
    print(f" + {p}")
print("\nCONS:")
for c in explanation.get("cons", []):
    print(f" - {c}")
    
print("\nSKILL EVIDENCE:")
for ev in explanation.get("skill_evidence", []):
    print(f" • {ev.get('skill')} -> {ev.get('source')} ({ev.get('file_or_commit')})")

--- STEP 6: EXPLANATION ENGINE ---

Calling Gemini to explain the score...

✅ Explanation Generated:

SUMMARY:
The candidate's score of 0.52 reflects a strong resume-based profile that is not fully supported by the provided code samples. While the experience signal is high due to relevant internship work, the low verification ratio suggests a lack of evidence in the repositories for the specific technical skills required for the role.

PROS:
 + Strong professional experience in AI research, specifically in multimodal vision-language frameworks and Graph Neural Networks.
 + Demonstrated ability to optimize legacy code, evidenced by the 10x speedup achieved by migrating Pandas logic to PyTorch.
 + High semantic alignment between resume experience and the AI research intern role requirements.

CONS:
 - Significant discrepancy between claimed skills and actual code repository content; most listed skills (PyTorch, Python, Transformers) were not found in the provided repositories.
 - Low ver